# Alkahest 0.8B Two-Stage Export

Runs the heavy post-training pipeline on Kaggle: candidate merge matrix, T4 text smoke, q4 ONNX export, package validation, and private Hugging Face upload when `HF_TOKEN` is available as a Kaggle secret.

In [ ]:
import os, platform, shutil
from pathlib import Path

print('python_platform=', platform.platform())
print('working_disk_free_gb=', round(shutil.disk_usage('/kaggle/working').free / 1024**3, 2))
print('input_dirs=', [str(p) for p in Path('/kaggle/input').glob('*')])
try:
    import torch
    print('torch=', torch.__version__)
    print('cuda_available=', torch.cuda.is_available())
    print('gpu_count=', torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        major, minor = torch.cuda.get_device_capability(i)
        print(f'gpu_{i}=', props.name, round(props.total_memory / 1024**3, 2), 'GiB', f'sm_{major}{minor}')
    if torch.cuda.is_available():
        major, minor = torch.cuda.get_device_capability(0)
        if major < 7:
            raise RuntimeError('Kaggle assigned a pre-Volta GPU. Push with --accelerator NvidiaTeslaT4.')
except Exception as exc:
    print('torch_probe_error=', repr(exc))
    if 'pre-Volta' in str(exc):
        raise

In [ ]:
import os, subprocess, sys

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
try:
    from kaggle_secrets import UserSecretsClient
    secret_token = UserSecretsClient().get_secret('HF_TOKEN')
    if secret_token and not os.environ.get('HF_TOKEN'):
        os.environ['HF_TOKEN'] = secret_token
    if secret_token and not os.environ.get('HUGGING_FACE_HUB_TOKEN'):
        os.environ['HUGGING_FACE_HUB_TOKEN'] = secret_token
    print('hf_secret_loaded=', bool(secret_token))
except Exception as exc:
    print('hf_secret_loaded=', False, type(exc).__name__)


packages = [
    'git+https://github.com/huggingface/transformers.git@c472755e79aac54d675845bff5e5c821c21260af',
    'accelerate>=1.13.0',
    'huggingface_hub[cli]>=1.5.0',
    'hf_transfer>=0.1.9',
    'safetensors>=0.7.0',
    'onnx>=1.19.0',
    'onnx-ir>=0.1.12',
    'onnxruntime>=1.23.0',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])
import transformers, huggingface_hub, onnx, onnxruntime
print('transformers=', transformers.__version__)
print('huggingface_hub=', huggingface_hub.__version__)
print('onnx=', onnx.__version__)
print('onnxruntime=', onnxruntime.__version__)
print('hf_token_present=', bool(os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')))

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/alkahest-ai/heretic-to-onnx.git'
REPO_REF = 'codex/kaggle-heretic-2b-run'
REPO_DIR = Path('/kaggle/working/heretic-to-onnx')

if REPO_DIR.exists():
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_REF])
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'checkout', REPO_REF])
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'])
else:
    subprocess.check_call(['git', 'clone', '--branch', REPO_REF, '--depth', '1', REPO_URL, str(REPO_DIR)])

print('repo=', REPO_DIR)
print('head=', subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip())

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_DIR = Path('/kaggle/working/heretic-to-onnx')
WORK_DIR = Path('/kaggle/working/alkahest-08b-two-stage-export')
cmd = [
    sys.executable,
    str(REPO_DIR / 'scripts/kaggle_alkahest_two_stage_export.py'),
    '--work-dir', str(WORK_DIR),
    '--artifact-name', os.environ.get('TWO_STAGE_ARTIFACT_NAME', 'alkahest-08b-two-stage-sft'),
    '--repo-prefix', os.environ.get('TWO_STAGE_REPO_PREFIX', 'thomasjvu/alkahest-0.8b-heretic-rp-sft-two-stage'),
    '--source-model-id', os.environ.get('TWO_STAGE_SOURCE_MODEL_ID', 'thomasjvu/alkahest-0.8b-heretic-merged'),
    '--template-model-id', os.environ.get('TWO_STAGE_TEMPLATE_MODEL_ID', 'onnx-community/Qwen3.5-0.8B-ONNX-OPT'),
    '--qwen-base-model-id', os.environ.get('TWO_STAGE_QWEN_BASE_MODEL_ID', 'Qwen/Qwen3.5-0.8B'),
    '--max-selected', os.environ.get('TWO_STAGE_MAX_SELECTED', '2'),
    '--max-new-tokens', os.environ.get('TWO_STAGE_MAX_NEW_TOKENS', '96'),
    '--temperature', os.environ.get('TWO_STAGE_TEMPERATURE', '0.2'),
]
if os.environ.get('TWO_STAGE_CANDIDATE_NAMES'):
    cmd.extend(['--candidate-names', os.environ['TWO_STAGE_CANDIDATE_NAMES']])
if os.environ.get('TWO_STAGE_SELECTED_CANDIDATES'):
    cmd.extend(['--selected-candidates', os.environ['TWO_STAGE_SELECTED_CANDIDATES']])
if os.environ.get('TWO_STAGE_NO_BASELINE_COMPARE') == '1':
    cmd.append('--no-compare-baseline')
if os.environ.get('TWO_STAGE_NO_UPLOAD') == '1':
    cmd.append('--no-upload')
if os.environ.get('TWO_STAGE_NO_EXPORT') == '1':
    cmd.append('--no-export')
if os.environ.get('TWO_STAGE_PUBLIC') == '1':
    cmd.append('--no-private')
print('running=', ' '.join(cmd))
subprocess.check_call(cmd)

In [ ]:
from pathlib import Path
import json

report_path = Path('/kaggle/working/alkahest-08b-two-stage-export/post-training-export-report.json')
print('report_path=', report_path)
if report_path.exists():
    report = json.loads(report_path.read_text())
    print(json.dumps({
        'ok': report.get('ok'),
        'selected': [item.get('name') for item in report.get('selected', [])],
        'exports': [item.get('repo_id') for item in report.get('exports', [])],
        'uploads': [item.get('upload', {}) for item in report.get('exports', [])],
    }, indent=2))
else:
    raise FileNotFoundError(report_path)